# 异常检测与自动干预

| 功能             | 行为                        |
| :------------- | :------------------------ |
| **NaN/Inf 检测** | 自动检测 loss/梯度中的非法数值，触发断点保存 |
| **梯度熔断**       | 梯度范数 > 阈值 时丢弃该 step        |
| **KL 熔断**      | KL 散度 > 阈值时停止更新或增大惩罚      |
| **长度熔断**       | 生成序列超过 max\_length 时强制截断  |


# 四大异常检测机制详解
## NaN/Inf 检测 —— 数值安全的"防火墙"

### 检测对象：Loss、Gradient、Logits、Ratio 等任何浮点张量

### 触发条件：

In [ ]:
torch.isnan(tensor).any()  # 存在 NaN
torch.isinf(tensor).any()  # 存在 Inf

### 根因分析：

| 异常位置           | 可能原因                                              |
| :------------- | :------------------------------------------------ |
| `loss` 为 NaN   | 重要性采样比率 $\rho$ 爆炸（旧策略概率 $\to$ 0）                  |
| `grad` 为 Inf   | 反向传播中某层梯度未裁剪，指数级放大                                |
| `logits` 为 Inf | 输入序列过长，注意力分数溢出 FP16 范围                            |
| `ratio` 为 Inf  | `old_log_prob` 被数值截断为 $-\infty$，导致 $\exp(\infty)$ |


### 干预行为：
- 立即终止当前 step 的反向传播
- 保存诊断快照：当前输入、模型状态、中间激活值
- 回滚参数到上一个稳定 checkpoint
- 可选：发送告警通知

## 梯度熔断 —— 防止"一步踏空"

### 原理：
- 当单步梯度范数超过阈值时，丢弃该 step 的全部梯度，不执行 optimizer.step()。

$if ∥g∥_2 > grad_breaker:skip update$

### 为什么不是裁剪而是丢弃？

| 场景         | 裁剪（clip）         | 熔断（drop）        |
| :--------- | :--------------- | :-------------- |
| 梯度范数 10    | 缩放到 1.0，正常更新     | 不适用             |
| 梯度范数 10000 | 缩放到 1.0，但方向可能已扭曲 | **直接丢弃**，避免污染模型 |
| 出现频率       | 偶发               | 连续出现说明系统性问题     |

熔断是最后一道防线，用于处理裁剪无法挽救的极端异常。

## KL 熔断 —— 防止策略"离家出走"
### 原理：
- 监控当前策略与参考模型（SFT）的 KL 散度。当 KL 超过阈值时：

| 策略       | 行为                               |
| :------- | :------------------------------- |
| **软熔断**  | 增大 KL 惩罚系数 $\beta$，强制拉回归         |
| **硬熔断**  | 直接停止该 step 的 Actor 更新，只更新 Critic |
| **紧急熔断** | 回退到上一个 KL 合格的 checkpoint         |

### 为什么 KL 飙升很危险？
- 策略分布剧烈偏移 → 重要性采样比率 ρ  极端化 → 梯度方差爆炸
- 模型学会奖励黑客（Reward Hacking）路径，偏离通用语言能力
- 后续生成的样本质量骤降，进入恶性循环

## 长度熔断 —— 打断"无限循环"
### 原理：
- 在生成阶段，如果序列长度达到 max_length 仍未输出 EOS，强制截断并追加 EOS 标记。

$if ∣y∣≥L_{max}:y←y[:L_{max}]+[EOS]$

### 为什么需要熔断而非自然停止？
- 没有熔断：模型可能无限重复某个 token（如 "..." 或 "the the the..."）
- 无限序列 → 显存 OOM → 训练崩溃
- 即使不 OOM，过长序列的反向传播计算量也 unacceptable

# 完整异常检测与干预系统


In [ ]:
import torch
import torch.nn as nn
import torch.distributed as dist
import os
import json
from datetime import datetime
from collections import defaultdict

class AnomalyDetector:
    """
    verl 风格的异常检测与自动干预系统
    覆盖：NaN/Inf 检测、梯度熔断、KL 熔断、长度熔断
    """
    def __init__(self, 
                 grad_breaker=1000.0,
                 kl_breaker=0.5,
                 kl_warning=0.1,
                 max_length=4096,
                 enable_nan_check=True,
                 enable_grad_breaker=True,
                 enable_kl_breaker=True,
                 enable_length_breaker=True,
                 checkpoint_dir='./emergency_ckpt'):
        
        self.grad_breaker = grad_breaker
        self.kl_breaker = kl_breaker
        self.kl_warning = kl_warning
        self.max_length = max_length
        self.enable_flags = {
            'nan': enable_nan_check,
            'grad': enable_grad_breaker,
            'kl': enable_kl_breaker,
            'length': enable_length_breaker,
        }
        self.checkpoint_dir = checkpoint_dir
        os.makedirs(checkpoint_dir, exist_ok=True)
        
        # 统计
        self.stats = defaultdict(int)
        self.kl_penalty_multiplier = 1.0  # 动态 KL 惩罚系数
    
    # ==================== 1. NaN/Inf 检测 ====================
    def check_nan_inf(self, **tensors):
        """
        检测所有传入张量中的 NaN/Inf
        
        Args:
            **tensors: 命名张量，如 loss=loss, ratio=ratio, grad_norm=grad_norm
        """
        if not self.enable_flags['nan']:
            return {'has_anomaly': False}
        
        anomalies = []
        for name, tensor in tensors.items():
            if tensor is None:
                continue
            
            # 统一处理标量和张量
            if isinstance(tensor, torch.Tensor):
                has_nan = torch.isnan(tensor).any().item()
                has_inf = torch.isinf(tensor).any().item()
            else:
                has_nan = math.isnan(tensor) if isinstance(tensor, float) else False
                has_inf = math.isinf(tensor) if isinstance(tensor, float) else False
            
            if has_nan:
                anomalies.append(f"{name}: NaN detected")
                self.stats['nan_count'] += 1
            if has_inf:
                anomalies.append(f"{name}: Inf detected")
                self.stats['inf_count'] += 1
        
        if anomalies:
            self._save_diagnostic_snapshot(tensors, anomalies)
        
        return {
            'has_anomaly': len(anomalies) > 0,
            'anomalies': anomalies,
        }
    
    def _save_diagnostic_snapshot(self, tensors, anomalies):
        """保存诊断快照用于事后分析"""
        timestamp = datetime.now().strftime("%Y%m%d_%H%M%S")
        snapshot_path = os.path.join(self.checkpoint_dir, f"snapshot_{timestamp}.json")
        
        snapshot = {
            'time': timestamp,
            'anomalies': anomalies,
            'tensor_shapes': {k: str(v.shape) if isinstance(v, torch.Tensor) else str(type(v)) 
                            for k, v in tensors.items()},
            'stats': dict(self.stats),
        }
        
        with open(snapshot_path, 'w') as f:
            json.dump(snapshot, f, indent=2)
        
        print(f"🚨 诊断快照已保存: {snapshot_path}")
    
    # ==================== 2. 梯度熔断 ====================
    def check_grad_breaker(self, model):
        """
        检查梯度范数是否超过熔断阈值
        返回：是否应丢弃该 step
        """
        if not self.enable_flags['grad']:
            return {'should_drop': False}
        
        total_norm = 0.0
        for p in model.parameters():
            if p.grad is not None:
                param_norm = p.grad.data.norm(2).item()
                total_norm += param_norm ** 2
        
        total_norm = total_norm ** 0.5
        
        should_drop = total_norm > self.grad_breaker
        
        if should_drop:
            self.stats['grad_breaker_triggered'] += 1
            print(f"🚨 梯度熔断触发！grad_norm={total_norm:.2f} > {self.grad_breaker}")
            
            # 清空梯度，防止误更新
            for p in model.parameters():
                if p.grad is not None:
                    p.grad.zero_()
        
        return {
            'grad_norm': total_norm,
            'should_drop': should_drop,
            'threshold': self.grad_breaker,
        }
    
    # ==================== 3. KL 熔断 ====================
    def check_kl_breaker(self, kl_div):
        """
        检查 KL 散度，动态调整惩罚系数或熔断
        
        Args:
            kl_div: 当前策略与参考模型的 KL 散度（标量）
        """
        if not self.enable_flags['kl']:
            return {'action': 'continue'}
        
        action = 'continue'
        
        if kl_div > self.kl_breaker:
            # 硬熔断：停止 Actor 更新
            action = 'hard_break'
            self.stats['kl_hard_break_triggered'] += 1
            print(f"🚨 KL 硬熔断！kl={kl_div:.4f} > {self.kl_breaker}，停止 Actor 更新")
            
        elif kl_div > self.kl_warning:
            # 软熔断：增大 KL 惩罚
            action = 'soft_break'
            self.kl_penalty_multiplier = min(self.kl_penalty_multiplier * 1.5, 10.0)
            self.stats['kl_soft_break_triggered'] += 1
            print(f"⚠️ KL 软熔断！kl={kl_div:.4f}，增大 KL 惩罚系数至 {self.kl_penalty_multiplier:.2f}")
            
        else:
            # 正常：逐步恢复惩罚系数
            self.kl_penalty_multiplier = max(self.kl_penalty_multiplier * 0.95, 1.0)
        
        return {
            'kl_div': kl_div,
            'action': action,
            'penalty_multiplier': self.kl_penalty_multiplier,
        }
    
    # ==================== 4. 长度熔断 ====================
    def check_length_breaker(self, generated_ids, eos_token_id):
        """
        检查并截断超长生成序列
        
        Args:
            generated_ids: [B, T] 当前生成的 token IDs
            eos_token_id: 结束符 ID
        
        Returns:
            truncated_ids: 截断后的序列
            truncation_stats: 截断统计
        """
        if not self.enable_flags['length']:
            return {'truncated_ids': generated_ids, 'num_truncated': 0}
        
        B, T = generated_ids.shape
        truncated = generated_ids.clone()
        num_truncated = 0
        
        for i in range(B):
            # 找第一个 EOS 位置
            seq = generated_ids[i].tolist()
            has_eos = eos_token_id in seq
            
            if not has_eos and T >= self.max_length:
                # 未找到 EOS 且达到长度上限 → 强制截断
                truncated[i, self.max_length - 1] = eos_token_id
                truncated[i, self.max_length:] = 0  # pad
                num_truncated += 1
        
        if num_truncated > 0:
            self.stats['length_breaker_triggered'] += num_truncated
            print(f"⚠️ 长度熔断：{num_truncated}/{B} 个样本被强制截断至 {self.max_length}")
        
        return {
            'truncated_ids': truncated,
            'num_truncated': num_truncated,
            'max_length': self.max_length,
        }
    
    # ==================== 综合诊断 ====================
    def should_skip_step(self, nan_inf_result, grad_result, kl_result):
        """
        综合判断是否应跳过当前 step 的更新
        """
        reasons = []
        
        if nan_inf_result.get('has_anomaly'):
            reasons.append("NaN/Inf detected")
        
        if grad_result.get('should_drop'):
            reasons.append("Gradient breaker triggered")
        
        if kl_result.get('action') == 'hard_break':
            reasons.append("KL hard breaker triggered")
        
        should_skip = len(reasons) > 0
        
        if should_skip:
            self.stats['skipped_steps'] += 1
        
        return {
            'should_skip': should_skip,
            'reasons': reasons,
        }
    
    def get_summary(self):
        """获取异常统计摘要"""
        total_steps = sum([
            self.stats['nan_count'],
            self.stats['grad_breaker_triggered'],
            self.stats['kl_hard_break_triggered'],
            self.stats['length_breaker_triggered'],
        ])
        
        return {
            'total_anomalies': total_steps,
            'nan_count': self.stats['nan_count'],
            'grad_breaker_count': self.stats['grad_breaker_triggered'],
            'kl_hard_count': self.stats['kl_hard_break_triggered'],
            'kl_soft_count': self.stats['kl_soft_break_triggered'],
            'length_breaker_count': self.stats['length_breaker_triggered'],
            'skipped_steps': self.stats['skipped_steps'],
        }


# ==================== 使用示例 ====================

print("=" * 70)
print("异常检测与自动干预系统演示")
print("=" * 70)

# 初始化检测器
detector = AnomalyDetector(
    grad_breaker=100.0,      # 设低一点方便演示
    kl_breaker=0.5,
    kl_warning=0.1,
    max_length=50,
)

# 模拟模型
model = nn.Linear(128, 64)
optimizer = torch.optim.Adam(model.parameters(), lr=1e-3)

for step in range(10):
    print(f"\n{'-'*50}")
    print(f"Step {step}")
    print(f"{'-'*50}")
    
    # 模拟前向
    x = torch.randn(4, 128)
    out = model(x)
    loss = out.mean()
    
    # 模拟 KL 散度（逐步增长）
    kl_div = 0.05 + step * 0.08  # step 6 超过 0.5
    
    # 模拟生成（长度逐步增长）
    gen_len = 30 + step * 5
    generated = torch.randint(0, 1000, (4, gen_len))
    
    # ========== 1. 长度熔断（生成阶段）==========
    length_result = detector.check_length_breaker(generated, eos_token_id=2)
    generated = length_result['truncated_ids']
    
    # ========== 2. NaN/Inf 检测 ==========
    # 模拟 step 3 出现 NaN
    if step == 3:
        loss = torch.tensor(float('nan'))
    
    nan_result = detector.check_nan_inf(loss=loss, kl_div=kl_div)
    
    # ========== 3. KL 熔断 ==========
    kl_result = detector.check_kl_breaker(kl_div)
    
    # 如果 NaN 或 KL 硬熔断，跳过反向传播
    if nan_result['has_anomaly'] or kl_result['action'] == 'hard_break':
        print(f"  ⏭️  跳过反向传播")
        continue
    
    # ========== 4. 反向传播 ==========
    optimizer.zero_grad()
    loss.backward()
    
    # ========== 5. 梯度熔断 ==========
    # 模拟 step 5 梯度爆炸
    if step == 5:
        for p in model.parameters():
            if p.grad is not None:
                p.grad *= 500.0
    
    grad_result = detector.check_grad_breaker(model)
    
    # ========== 6. 综合判断是否跳过更新 ==========
    skip_decision = detector.should_skip_step(nan_result, grad_result, kl_result)
    
    if skip_decision['should_skip']:
        print(f"  ⏭️  跳过 optimizer.step()，原因: {skip_decision['reasons']}")
        optimizer.zero_grad()
    else:
        # 应用动态 KL 惩罚
        adjusted_loss = loss + kl_result.get('penalty_multiplier', 1.0) * kl_div
        
        # 正常更新
        torch.nn.utils.clip_grad_norm_(model.parameters(), 1.0)
        optimizer.step()
        print(f"  ✅ 正常更新 (KL penalty: {kl_result.get('penalty_multiplier', 1.0):.2f}x)")

# 最终统计
print(f"\n{'='*70}")
print("异常统计摘要")
print(f"{'='*70}")
summary = detector.get_summary()
for k, v in summary.items():
    print(f"  {k}: {v}")
print(f"{'='*70}")

# 在 RL 训练循环中的标准集成

In [ ]:
def train_step_with_anomaly_detection(
    model, optimizer, detector, batch, ref_model, 
    eos_token_id, max_length
):
    """
    标准 RL 训练 step，集成全部异常检测
    """
    # ========== 阶段 1：生成 ==========
    with torch.no_grad():
        generated = model.generate(batch['prompts'], max_length=max_length)
        
        # 长度熔断
        length_result = detector.check_length_breaker(generated, eos_token_id)
        generated = length_result['truncated_ids']
        
        # 获取参考模型概率（用于 KL 计算）
        ref_logits = ref_model(generated)
        ref_log_probs = F.log_softmax(ref_logits, dim=-1)
    
    # ========== 阶段 2：前向 + 计算损失 ==========
    logits = model(generated)
    log_probs = F.log_softmax(logits, dim=-1)
    
    # 计算 PPO 损失
    ratio = torch.exp(log_probs - old_log_probs)
    advantages = batch['advantages']
    
    surr1 = ratio * advantages
    surr2 = torch.clamp(ratio, 0.8, 1.2) * advantages
    policy_loss = -torch.min(surr1, surr2).mean()
    
    # 计算 KL
    kl_div = (ref_log_probs - log_probs).mean()
    
    # 总损失（含动态 KL 惩罚）
    total_loss = policy_loss + detector.kl_penalty_multiplier * kl_div
    
    # ========== 阶段 3：NaN/Inf 预检测 ==========
    nan_result = detector.check_nan_inf(
        loss=total_loss,
        ratio=ratio,
        kl_div=kl_div,
    )
    
    if nan_result['has_anomaly']:
        return {'status': 'skipped', 'reason': 'NaN/Inf', 'step_type': 'pre_backward'}
    
    # ========== 阶段 4：反向传播 ==========
    optimizer.zero_grad()
    total_loss.backward()
    
    # ========== 阶段 5：梯度熔断 ==========
    grad_result = detector.check_grad_breaker(model)
    
    # ========== 阶段 6：KL 熔断 ==========
    kl_result = detector.check_kl_breaker(kl_div.item())
    
    # ========== 阶段 7：综合决策 ==========
    skip = detector.should_skip_step(nan_result, grad_result, kl_result)
    
    if skip['should_skip']:
        optimizer.zero_grad()
        return {
            'status': 'skipped',
            'reason': skip['reasons'],
            'grad_norm': grad_result.get('grad_norm'),
            'kl_div': kl_div.item(),
        }
    
    # ========== 阶段 8：正常更新 ==========
    torch.nn.utils.clip_grad_norm_(model.parameters(), 1.0)
    optimizer.step()
    
    return {
        'status': 'success',
        'loss': total_loss.item(),
        'kl_div': kl_div.item(),
        'grad_norm': grad_result.get('grad_norm'),
    }

# 异常检测的优先级与联动

In [ ]:
异常处理优先级（从高到低）：

1. NaN/Inf 检测    → 立即跳过，保存快照（最致命）
        ↓
2. 梯度熔断        → 清空梯度，跳过更新（防止参数污染）
        ↓
3. KL 硬熔断       → 停止 Actor 更新（防止策略偏移）
        ↓
4. KL 软熔断       → 增大惩罚，继续更新（预警状态）
        ↓
5. 长度熔断        → 截断序列，继续生成（常规保护）

# 生产环境配置建议

In [ ]:
# verl 配置示例
anomaly_detection:
  enabled: true
  
  nan_inf:
    enable: true
    save_snapshot: true
    
  grad_breaker:
    enable: true
    threshold: 1000.0
    
  kl_breaker:
    enable: true
    hard_threshold: 0.5      # 停止 Actor 更新
    soft_threshold: 0.1      # 增大惩罚
    penalty_max: 10.0       # 惩罚系数上限
    
  length_breaker:
    enable: true
    max_length: 4096
    force_eos: true         # 强制追加 EOS

# 一句话总结
- 异常检测与自动干预是 RL 训练的"安全气囊"：NaN/Inf 检测防止数值灾难，梯度熔断防止参数跳崖，KL 熔断防止策略离家出走，长度熔断防止无限循环。四层防护缺一不可，它们共同确保训练在"撞墙"前自动刹车，而不是带着全车的参数一起坠入深渊。
